In [ ]:
import torch
import torch.nn.functional as F

def compute_adaptive_mask(video_3: torch.Tensor, video_20: torch.Tensor,
                          voxel_size: int = 2, patch_size: int = 10):
    """
    Args:
        video_3: Tensor of shape (3, H, W) – frames T-1, T, T+1
        video_20: Tensor of shape (20, H, W) – for temporal baseline
        voxel_size: e.g., 2 for 2×2 voxels
        patch_size: e.g., 10 for 10×10 context patch
    
    Returns:
        adaptive_mask: (H//2, W//2) boolean tensor
    """
    T, H, W = video_3.shape
    _, H20, W20 = video_20.shape
    assert H == H20 and W == W20, "Spatial size mismatch"

    center_frame = video_3[1]
    mask_h, mask_w = H // voxel_size, W // voxel_size
    adaptive_mask = torch.zeros((mask_h, mask_w), dtype=torch.bool)

    pad = patch_size // 2
    padded_center = F.pad(center_frame, (pad, pad, pad, pad), mode='reflect')
    padded_video20 = F.pad(video_20, (pad, pad, pad, pad), mode='reflect')

    for i in range(mask_h):
        for j in range(mask_w):
            cx, cy = i * voxel_size + voxel_size // 2, j * voxel_size + voxel_size // 2
            # Extract 10×10 patch from center frame
            patch = padded_center[cx:cx + patch_size, cy:cy + patch_size]
            spatial_mean = patch.mean()

            # Extract same patch over 20 frames
            temp_patch = padded_video20[:, cx:cx + patch_size, cy:cy + patch_size]
            temporal_mean = temp_patch.mean()

            # If spike-like → use spatial context, else temporal
            adaptive_mask[i, j] = spatial_mean > temporal_mean

    return adaptive_mask

def create_adaptive_downsampled_images(video_3: torch.Tensor,
                                       video_20: torch.Tensor,
                                       adaptive_mask: torch.Tensor,
                                       voxel_size: int = 2,
                                       patch_size: int = 10):
    """
    Generate downsampled input and target images using adaptive spatial/temporal sampling.

    Args:
        video_3: Tensor of shape (3, H, W) – frames [t-1, t, t+1]
        video_20: Tensor of shape (20, H, W) – for temporal patch baseline
        adaptive_mask: Tensor of shape (H//2, W//2), bool – spatial (True) or temporal (False)
        voxel_size: e.g. 2 for 2×2 voxel
        patch_size: e.g. 10 for 10×10 patch for activity decision

    Returns:
        input_image: Downsampled input image (H//2, W//2)
        target_image: Downsampled target image (H//2, W//2)
    """
    T, H, W = video_3.shape
    assert video_20.shape[1:] == (H, W)

    center_frame = video_3[1]
    prev_frame = video_3[0]
    next_frame = video_3[2]

    pad = patch_size // 2
    padded_video_3 = F.pad(video_3, (pad, pad, pad, pad), mode='reflect')
    padded_video_20 = F.pad(video_20, (pad, pad, pad, pad), mode='reflect')

    down_H, down_W = H // voxel_size, W // voxel_size
    input_img = torch.zeros((down_H, down_W))
    target_img = torch.zeros((down_H, down_W))

    for i in range(down_H):
        for j in range(down_W):
            x, y = i * voxel_size, j * voxel_size
            patch = center_frame[x:x+2, y:y+2]

            if adaptive_mask[i, j]:  # Spatial sampling
                diag = (patch[0, 0] + patch[1, 1]) / 2
                anti_diag = (patch[0, 1] + patch[1, 0]) / 2
                input_img[i, j] = diag
                target_img[i, j] = anti_diag
            else:  # Temporal sampling
                cx, cy = x + 1, y + 1
                i_start = max(cx - patch_size // 2, 0)
                i_end = min(i_start + patch_size, H)
                j_start = max(cy - patch_size // 2, 0)
                j_end = min(j_start + patch_size, W)

                # Get patch averages
                prev_patch = prev_frame[i_start:i_end, j_start:j_end].mean()
                next_patch = next_frame[i_start:i_end, j_start:j_end].mean()
                temporal_baseline = video_20[:, i_start:i_end, j_start:j_end].mean()

                # Pick temporally quieter neighbor
                if prev_patch < next_patch:
                    target_pixel = prev_frame[x + 1, y + 1]
                else:
                    target_pixel = next_frame[x + 1, y + 1]

                center_pixel = center_frame[x + 1, y + 1]
                input_img[i, j] = center_pixel
                target_img[i, j] = target_pixel

    return input_img, target_img